# 06 — Interpretability Analysis

Feature importance from GBDTs, attention patterns from Transformers, and model behavior comparison.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.registry import load_dataset
from src.data.preprocessing import preprocess_for_gbdt, preprocess_for_deep_learning
from src.models.factory import create_model

# Try to import SHAP; skip gracefully if not installed
try:
    import shap
    SHAP_AVAILABLE = True
    print("SHAP is available — SHAP analyses will run.")
except ImportError:
    SHAP_AVAILABLE = False
    print("WARNING: 'shap' package not installed. SHAP sections will be skipped.")
    print("Install with: pip install shap")

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'savefig.dpi': 300})

os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

# Reproducibility
import random
random.seed(42)
np.random.seed(42)

In [ ]:
# Load multiple datasets for interpretability analysis.
# We select one small, one medium, and one large classification dataset.
# 'adult' is the primary dataset; fall back to whatever is available.

from sklearn.model_selection import train_test_split

# Datasets to analyse: (name, desired task type hint)
# We try these in order and keep whichever load successfully.
DATASETS_TO_TRY = ['adult', 'covertype', 'bank_marketing', 'higgs', 'otto_group']
REGRESSION_DATASETS = ['california_housing', 'diamonds', 'abalone']

# Primary (classification) dataset
PRIMARY_DATASET = None
PRIMARY_SPLITS = {}

for ds_name in DATASETS_TO_TRY:
    try:
        X, y, info = load_dataset(ds_name)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42,
            stratify=(y if info.task_type in ('binary', 'multiclass') else None)
        )
        PRIMARY_DATASET = ds_name
        PRIMARY_SPLITS[ds_name] = {
            'X': X, 'y': y, 'info': info,
            'X_train': X_train, 'X_test': X_test,
            'y_train': y_train, 'y_test': y_test,
        }
        print(f"Loaded primary dataset: {ds_name}  "
              f"({len(X)} rows, {X.shape[1]} features, task={info.task_type})")
        break
    except Exception as e:
        print(f"  Skipping {ds_name}: {e}")

if PRIMARY_DATASET is None:
    raise RuntimeError("Could not load any classification dataset. "
                       "Run scripts/download_data.py first.")

# Extra datasets for multi-dataset analyses (at least 3 total)
EXTRA_DATASETS = []
for ds_name in DATASETS_TO_TRY + REGRESSION_DATASETS:
    if ds_name == PRIMARY_DATASET:
        continue
    if len(EXTRA_DATASETS) >= 2:
        break
    try:
        X2, y2, info2 = load_dataset(ds_name)
        X2_tr, X2_te, y2_tr, y2_te = train_test_split(
            X2, y2, test_size=0.2, random_state=42,
            stratify=(y2 if info2.task_type in ('binary', 'multiclass') else None)
        )
        EXTRA_DATASETS.append(ds_name)
        PRIMARY_SPLITS[ds_name] = {
            'X': X2, 'y': y2, 'info': info2,
            'X_train': X2_tr, 'X_test': X2_te,
            'y_train': y2_tr, 'y_test': y2_te,
        }
        print(f"Loaded extra dataset:    {ds_name}  "
              f"({len(X2)} rows, {X2.shape[1]} features, task={info2.task_type})")
    except Exception as e:
        print(f"  Skipping {ds_name}: {e}")

ALL_DATASETS = [PRIMARY_DATASET] + EXTRA_DATASETS
print(f"\nDatasets for analysis: {ALL_DATASETS}")

In [ ]:
# XGBoost feature importance — primary dataset
# NOTE: preprocess_for_gbdt returns a PreprocessedData object with fields:
#   .X_train (preprocessed training features)
#   .X_val   (preprocessed validation/test features, same transform as train)
# The bug in the original notebook was using prep.X_test (non-existent field).
# Correct field name is prep.X_val when X_val is passed to preprocess_for_gbdt.

ds = PRIMARY_SPLITS[PRIMARY_DATASET]
info = ds['info']

# Preprocess: pass X_test as X_val so it gets the same transform
prep = preprocess_for_gbdt(ds['X_train'], info, X_val=ds['X_test'])
# prep.X_train  → transformed training set
# prep.X_val    → transformed test set (same pipeline, no leakage)

xgb_model = create_model('xgboost', info.task_type, info.n_classes)
xgb_model.fit(prep.X_train, ds['y_train'])

# Feature names after preprocessing (ordinal encoding keeps column names intact)
feature_names = info.num_columns + info.cat_columns
importance = xgb_model.model.feature_importances_

imp_df = pd.DataFrame({
    'feature': feature_names[:len(importance)],
    'importance': importance
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
imp_df.tail(20).plot.barh(x='feature', y='importance', ax=ax,
                           legend=False, color='steelblue')
ax.set_title(f'XGBoost Feature Importance — {PRIMARY_DATASET} (top 20)')
ax.set_xlabel('Importance (gain)')
plt.tight_layout()
plt.savefig('../results/figures/feature_importance_xgboost.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/figures/feature_importance_xgboost.pdf', bbox_inches='tight')
plt.show()

# Store the preprocessed data for later reuse (avoids double preprocessing)
PREP_CACHE = {PRIMARY_DATASET: {'prep': prep, 'y_train': ds['y_train'], 'y_test': ds['y_test']}}

In [ ]:
# Compare feature importance across all 3 GBDTs on the primary dataset
# Uses prep from the previous cell (already fitted on X_train, transforms X_test as X_val)

feature_names = info.num_columns + info.cat_columns
GBDT_MODELS = ['xgboost', 'lightgbm', 'catboost']

gbdt_importances = {}  # model_name -> normalized importance array

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, model_name in zip(axes, GBDT_MODELS):
    model = create_model(model_name, info.task_type, info.n_classes)
    model.fit(prep.X_train, ds['y_train'])

    importance = model.model.feature_importances_
    norm_importance = importance / importance.sum()
    gbdt_importances[model_name] = norm_importance

    imp_df = pd.DataFrame({
        'feature': feature_names[:len(norm_importance)],
        'importance': norm_importance,
    }).sort_values('importance', ascending=True).tail(15)

    imp_df.plot.barh(x='feature', y='importance', ax=ax, legend=False, color='steelblue')
    ax.set_title(f'{model_name.upper()} — Top 15 Features')
    ax.set_xlabel('Normalized Importance')

plt.suptitle(f'GBDT Feature Importance Comparison — {PRIMARY_DATASET}', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/feature_importance_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/figures/feature_importance_comparison.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Prediction agreement (Cohen's kappa) between models — extended to all model families
# Cohen's kappa measures agreement beyond chance for classification models.
# For regression, we discretize predictions into quantile bins before computing kappa.

from sklearn.metrics import cohen_kappa_score

model_families = {
    'xgboost': 'GBDT', 'lightgbm': 'GBDT', 'catboost': 'GBDT',
    'ft_transformer': 'DL', 'tabnet': 'DL', 'saint': 'DL',
    'tabm': 'DL', 'mlp': 'DL', 'realmlp': 'DL',
    'tabpfn': 'FM'
}

# Start with GBDTs (always available — no GPU required)
MODELS_TO_COMPARE = ['xgboost', 'lightgbm', 'catboost']

# Optionally add DL/FM models — they may be slow or unavailable without GPU
DL_MODELS_TO_TRY = ['tabnet', 'ft_transformer', 'tabpfn']

predictions = {}

# Reuse models already fitted above for GBDTs to avoid redundant training
for model_name in GBDT_MODELS:
    m = create_model(model_name, info.task_type, info.n_classes)
    m.fit(prep.X_train, ds['y_train'])
    # Use prep.X_val as the held-out test set (same transform, no leakage)
    predictions[model_name] = m.predict(prep.X_val)

# Try adding DL/FM models (skip if model unavailable or takes too long)
for model_name in DL_MODELS_TO_TRY:
    try:
        m = create_model(model_name, info.task_type, info.n_classes)
        m.fit(prep.X_train, ds['y_train'])
        predictions[model_name] = m.predict(prep.X_val)
        MODELS_TO_COMPARE.append(model_name)
        print(f"Added {model_name} to prediction agreement analysis.")
    except Exception as e:
        print(f"Skipping {model_name} (prediction agreement): {e}")

all_models = list(predictions.keys())
n_models = len(all_models)

# Build kappa matrix
kappa_matrix = pd.DataFrame(
    np.ones((n_models, n_models)),
    index=all_models, columns=all_models,
)
for i, m1 in enumerate(all_models):
    for j, m2 in enumerate(all_models):
        if i != j:
            try:
                kappa_matrix.loc[m1, m2] = cohen_kappa_score(
                    predictions[m1], predictions[m2]
                )
            except Exception:
                kappa_matrix.loc[m1, m2] = np.nan

plt.figure(figsize=(max(8, n_models + 2), max(6, n_models)))
sns.heatmap(
    kappa_matrix.astype(float), annot=True, fmt='.3f',
    cmap='YlGn', vmin=0.5, vmax=1.0,
    linewidths=0.5
)
plt.title(f"Prediction Agreement (Cohen's Kappa) — {PRIMARY_DATASET}\n"
          f"(1.0 = perfect agreement, 0.0 = chance-level)")
plt.tight_layout()
plt.savefig('../results/figures/prediction_agreement.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/figures/prediction_agreement.pdf', bbox_inches='tight')
plt.show()

print("\nMean kappa per model (how much it agrees with all others):")
mean_kappa = kappa_matrix.replace(1.0, np.nan).mean(axis=1).sort_values(ascending=False)
print(mean_kappa)

In [ ]:
# Feature importance stability: do GBDTs agree on which features matter most?
# We compare the top-K feature rankings across XGBoost, LightGBM, and CatBoost.
# High agreement = stable, robust feature importance signal.
# Low agreement = models use different information to make predictions.

from scipy.stats import spearmanr

feature_names = info.num_columns + info.cat_columns

# Build a DataFrame: rows = features, columns = models, values = normalized importance
imp_matrix = pd.DataFrame(index=feature_names)
for model_name, importance in gbdt_importances.items():
    col_name = model_name
    norm = importance / importance.sum()
    imp_matrix[col_name] = norm[:len(feature_names)]

# Rank correlation between all pairs of models
print("Spearman rank correlation of feature importances between GBDT models:")
model_list = list(gbdt_importances.keys())
corr_data = {}
for m1 in model_list:
    row = {}
    for m2 in model_list:
        rho, p = spearmanr(imp_matrix[m1].fillna(0), imp_matrix[m2].fillna(0))
        row[m2] = rho
    corr_data[m1] = row

corr_df = pd.DataFrame(corr_data)
print(corr_df.round(3))

# Heatmap of rank correlation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(corr_df.astype(float), annot=True, fmt='.3f',
            cmap='RdYlGn', vmin=-1, vmax=1, ax=axes[0],
            linewidths=0.5)
axes[0].set_title(f'Feature Importance Rank Correlation\n(Spearman ρ) — {PRIMARY_DATASET}')

# Top-K feature agreement: how many of the top-10 features are shared?
TOP_K = min(10, len(feature_names))
top_sets = {}
for model_name in model_list:
    top_sets[model_name] = set(
        imp_matrix[model_name].nlargest(TOP_K).index.tolist()
    )

# Overlap matrix
overlap = pd.DataFrame(index=model_list, columns=model_list, dtype=float)
for m1 in model_list:
    for m2 in model_list:
        overlap.loc[m1, m2] = len(top_sets[m1] & top_sets[m2]) / TOP_K

sns.heatmap(overlap.astype(float), annot=True, fmt='.2f',
            cmap='Blues', vmin=0, vmax=1, ax=axes[1],
            linewidths=0.5)
axes[1].set_title(f'Top-{TOP_K} Feature Overlap Fraction — {PRIMARY_DATASET}')

plt.tight_layout()
plt.savefig('../results/figures/feature_stability.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/figures/feature_stability.pdf', bbox_inches='tight')
plt.show()

# Print the consensus top features (in top-K for at least 2 out of 3 models)
all_top = [f for f in feature_names if
           sum(f in top_sets[m] for m in model_list) >= 2]
print(f"\nConsensus top features (top-{TOP_K} in ≥ 2/{len(model_list)} models): {all_top}")

In [ ]:
# Multi-dataset feature importance: train GBDTs on each dataset and compare top features.
# For each dataset we show the top-10 features by XGBoost importance.
# This section gracefully handles datasets with different feature sets.

N_TOP = 10

n_datasets = len(ALL_DATASETS)
fig, axes = plt.subplots(1, n_datasets, figsize=(7 * n_datasets, 7), squeeze=False)

for col_idx, ds_name in enumerate(ALL_DATASETS):
    ax = axes[0][col_idx]
    try:
        ds_data = PRIMARY_SPLITS[ds_name]
        ds_info = ds_data['info']

        ds_prep = preprocess_for_gbdt(ds_data['X_train'], ds_info, X_val=ds_data['X_test'])
        ds_feat_names = ds_info.num_columns + ds_info.cat_columns

        m = create_model('xgboost', ds_info.task_type, ds_info.n_classes)
        m.fit(ds_prep.X_train, ds_data['y_train'])

        imp = m.model.feature_importances_
        norm_imp = imp / imp.sum()

        imp_df = pd.DataFrame({
            'feature': ds_feat_names[:len(norm_imp)],
            'importance': norm_imp,
        }).sort_values('importance', ascending=True).tail(N_TOP)

        imp_df.plot.barh(x='feature', y='importance', ax=ax,
                          legend=False, color='steelblue')
        ax.set_title(f'XGBoost Top-{N_TOP}\n{ds_name}', fontsize=11)
        ax.set_xlabel('Norm. Importance')

    except Exception as e:
        ax.text(0.5, 0.5, f"Failed:\n{e}", ha='center', va='center',
                transform=ax.transAxes, fontsize=9)
        ax.set_title(ds_name)

plt.suptitle('XGBoost Feature Importance — Multi-Dataset Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/feature_importance_multi_dataset.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/figures/feature_importance_multi_dataset.pdf', bbox_inches='tight')
plt.show()

print("\nNote: Each dataset has a different feature set, so features are not directly comparable.")
print("The plots show that XGBoost learns dataset-specific important features rather than")
print("relying on a fixed set of signals — consistent with the literature on GBDT robustness.")

## Multi-Dataset Feature Importance Comparison

Do different model families agree on which features matter across different datasets?
Here we compare XGBoost feature importances across all loaded datasets to see whether the relative importance of features is consistent, and whether GBDT models give similar rankings on each dataset.

In [ ]:
# SHAP analysis using TreeExplainer for GBDTs
# Runs across ALL_DATASETS (up to 3) for richer comparisons.
# Requires: pip install shap

if not SHAP_AVAILABLE:
    print("SHAP not available — skipping this section.")
    print("Install with: pip install shap")
else:
    for ds_name in ALL_DATASETS:
        print(f"\n{'='*60}")
        print(f"SHAP analysis: {ds_name}")
        print('='*60)

        try:
            ds_data = PRIMARY_SPLITS[ds_name]
            ds_info = ds_data['info']

            # Preprocess for GBDTs — correct field is prep.X_val for test set
            ds_prep = preprocess_for_gbdt(
                ds_data['X_train'], ds_info, X_val=ds_data['X_test']
            )

            ds_feat_names = ds_info.num_columns + ds_info.cat_columns

            # Use XGBoost as the primary SHAP model
            shap_model = create_model('xgboost', ds_info.task_type, ds_info.n_classes)
            shap_model.fit(ds_prep.X_train, ds_data['y_train'])

            # Build TreeExplainer — works directly on the underlying XGBoost booster
            explainer = shap.TreeExplainer(shap_model.model)

            # Compute SHAP values on the test set (cap at 500 samples for speed)
            X_shap = ds_prep.X_val
            if hasattr(X_shap, 'values'):
                X_shap_arr = X_shap.values
            else:
                X_shap_arr = np.array(X_shap)

            n_shap = min(500, len(X_shap_arr))
            X_shap_sample = X_shap_arr[:n_shap]

            shap_values = explainer.shap_values(X_shap_sample)

            # For multi-class output, shap_values is a list; use first class or flatten
            if isinstance(shap_values, list):
                # Binary: shap_values[1] is for positive class
                shap_plot_values = shap_values[1] if len(shap_values) == 2 else shap_values[0]
            else:
                shap_plot_values = shap_values

            # Beeswarm summary plot
            fig_shap, ax_shap = plt.subplots(figsize=(10, max(6, min(len(ds_feat_names), 20) * 0.4 + 2)))
            shap.summary_plot(
                shap_plot_values,
                X_shap_sample,
                feature_names=ds_feat_names[:X_shap_sample.shape[1]],
                max_display=20,
                show=False,
                plot_type='dot',
            )
            plt.title(f'SHAP Beeswarm — XGBoost on {ds_name} (n={n_shap})', fontsize=13)
            plt.tight_layout()
            plt.savefig(f'../results/figures/shap_beeswarm_{ds_name}.png', dpi=150, bbox_inches='tight')
            plt.savefig(f'../results/figures/shap_beeswarm_{ds_name}.pdf', bbox_inches='tight')
            plt.show()

            # Bar chart of mean |SHAP| importance (easier to read in papers)
            mean_abs_shap = np.abs(shap_plot_values).mean(axis=0)
            shap_imp_df = pd.DataFrame({
                'feature': ds_feat_names[:len(mean_abs_shap)],
                'mean_abs_shap': mean_abs_shap,
            }).sort_values('mean_abs_shap', ascending=True).tail(15)

            fig2, ax2 = plt.subplots(figsize=(10, 6))
            shap_imp_df.plot.barh(x='feature', y='mean_abs_shap', ax=ax2,
                                   legend=False, color='tomato')
            ax2.set_title(f'SHAP Feature Importance (mean |SHAP|) — XGBoost / {ds_name}')
            ax2.set_xlabel('Mean |SHAP value|')
            plt.tight_layout()
            plt.savefig(f'../results/figures/shap_importance_{ds_name}.png', dpi=300, bbox_inches='tight')
            plt.savefig(f'../results/figures/shap_importance_{ds_name}.pdf', bbox_inches='tight')
            plt.show()

        except Exception as e:
            print(f"SHAP analysis failed for {ds_name}: {e}")

    # Note on DL SHAP
    print("""
--- DL / Foundation Model SHAP (Template) ---
For neural network models (FT-Transformer, TabNet, etc.) use:
  explainer = shap.DeepExplainer(model.model, background_tensor)
  shap_values = explainer.shap_values(X_test_tensor)
Or for any model (slow):
  explainer = shap.KernelExplainer(model.predict_proba, shap.sample(X_train, 100))
  shap_values = explainer.shap_values(X_test[:50])
DL SHAP requires careful tensor preparation and is best run on GPU.
""")

## SHAP Analysis

SHAP (SHapley Additive exPlanations) provides model-agnostic, game-theory-based feature attributions.

- **TreeExplainer**: Exact, fast SHAP values for tree-based models (XGBoost, LightGBM, CatBoost). O(TLD²) per instance.
- **DeepExplainer / GradientExplainer**: Approximate SHAP for neural networks (requires PyTorch/TF). Slower.
- **KernelExplainer**: Model-agnostic (works for any model) but very slow — use with a small background set.

We use `TreeExplainer` for GBDTs here. DL SHAP is noted as a template requiring GPU.

SHAP beeswarm plots show:
- Each dot = one sample
- X-axis = SHAP value (positive = pushed towards positive class)
- Color = feature value (red = high, blue = low)
- Features ordered by mean |SHAP| importance